<a href="https://colab.research.google.com/github/mariakhanoishi01-sketch/data-cleaning-project/blob/main/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall()

print("Files extracted")

import zipfile
# Extract ZIP

In [ ]:
import os

os.listdir()

In [ ]:
!pip install pandas numpy

In [ ]:
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports", exist_ok=True)

print("Folders created")

In [ ]:
import shutil

shutil.copy(
    "metadata.csv",
    "data/raw/metadata.csv"
)

print("Dataset moved")

In [ ]:
"""
==============================================================================
Antimicrobial Resistance Analysis and Prediction
Data Cleaning & Preprocessing Pipeline

Dataset Overview
----------------
- 3,786 bacterial isolates (rows) × 31 features (columns)
- Covers isolates from 65 countries across 5 continents (1979–2017)
- Contains phenotypic MIC (Minimum Inhibitory Concentration) values,
  binary S/R classifications, genomic group assignments, and metadata
- Antimicrobials: Azithromycin, Ciprofloxacin, Ceftriaxone,
                  Cefixime, Tetracycline, Penicillin

Key Data Quality Issues Found
------------------------------
1. Missing values: Heavy missingness in Tetracycline/Penicillin (~61%),
   moderate in Beta.lactamase (~49%), Year (~7%)
2. Censored MIC strings: e.g. '>256', '<=0.008', '>=64/>=512' in
   phenotypic antibiotic columns — not directly numeric
3. Mixed Beta.lactamase encoding: 'S'/'R' mixed with '0'/'1'/'2'
4. One row missing Country/Continent pair
5. Year stored as float64 — should be nullable integer
6. No true duplicate rows (confirmed)
7. Composite censored values: e.g. '>=64/>=512' require special parsing
==============================================================================
"""

import re
import sys
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

# Input / output paths (adjust as needed)
RAW_DATA_PATH   = Path("data/raw/metadata.csv")
CLEAN_DATA_PATH = Path("data/processed/metadata_cleaned.csv")
REPORT_PATH     = Path("reports/data_quality_report.txt")

# Antibiotic columns that contain raw phenotypic MIC strings
ANTIBIOTIC_COLS = [
    "Azithromycin", "Ciprofloxacin", "Ceftriaxone",
    "Cefixime",     "Tetracycline",  "Penicillin",
]

# Corresponding numeric MIC columns (already computed in source data)
MIC_NUM_COLS = [
    "azm_mic", "cip_mic", "cro_mic",
    "cfx_mic", "tet_mic", "pen_mic",
]

# Log2-transformed MIC columns
LOG2_MIC_COLS = [
    "log2_azm_mic", "log2_cip_mic", "log2_cro_mic",
    "log2_cfx_mic", "log2_tet_mic", "log2_pen_mic",
]

# Binary susceptibility/resistance columns (0 = susceptible, 1 = resistant)
SR_COLS = [
    "azm_sr", "cip_sr", "cro_sr",
    "cfx_sr", "tet_sr", "pen_sr",
]

# ---------------------------------------------------------------------------
# Logging Setup
# ---------------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------------------------

def parse_censored_mic(value: str) -> tuple[float | None, str | None]:
    """
    Parse a raw MIC string (which may be censored) into a numeric value
    and a censor flag.

    Examples
    --------
    '>256'          → (256.0, 'right')
    '<=0.008'       → (0.008, 'left')
    '>=64/>=512'    → (64.0,  'right')   # take the lower bound
    '0.125'         → (0.125,  None)     # exact measurement

    Parameters
    ----------
    value : str
        Raw MIC string from the source CSV.

    Returns
    -------
    numeric : float | None
        Parsed numeric MIC value (boundary value for censored data).
    censor_flag : str | None
        'right' for right-censored ('>','>='), 'left' for left-censored
        ('<','<='), None for exact measurements.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None, None

    value = str(value).strip()

    # Handle composite censored values like '>=64/>=512' — take first part
    if "/" in value:
        value = value.split("/")[0].strip()

    # Right-censored: '>' or '>='
    right_match = re.match(r"^>=?\s*(.+)$", value)
    if right_match:
        try:
            return float(right_match.group(1)), "right"
        except ValueError:
            return None, None

    # Left-censored: '<' or '<='
    left_match = re.match(r"^<=?\s*(.+)$", value)
    if left_match:
        try:
            return float(left_match.group(1)), "left"
        except ValueError:
            return None, None

    # Exact numeric value (may have whitespace)
    try:
        return float(value), None
    except ValueError:
        return None, None


def standardise_beta_lactamase(value: str) -> str | None:
    """
    Harmonise the mixed Beta.lactamase encoding.

    Observed values:
        'S'  → susceptible (no beta-lactamase)  → map to '0' / 'Negative'
        'R'  → resistant   (beta-lactamase +ve) → map to '1+' / 'Positive'
        '0'  → negative    (consistent with S)
        '1'  → low-level producer
        '2'  → high-level producer
        NaN  → unknown

    We standardise to a clean categorical: ['Negative','Low','High','Unknown']

    Parameters
    ----------
    value : str
        Raw Beta.lactamase cell value.

    Returns
    -------
    str | None
        Standardised label.
    """
    if pd.isna(value):
        return "Unknown"

    mapping = {
        "S": "Negative",   # susceptible → no beta-lactamase activity
        "0": "Negative",
        "R": "Low",        # resistant but level unspecified → conservative
        "1": "Low",
        "2": "High",
    }
    return mapping.get(str(value).strip(), "Unknown")


def compute_log2_mic(mic_value: float | None) -> float | None:
    """
    Safely compute log2(MIC), returning None for non-positive or missing inputs.

    Parameters
    ----------
    mic_value : float | None

    Returns
    -------
    float | None
    """
    if mic_value is None or np.isnan(mic_value) or mic_value <= 0:
        return None
    return np.log2(mic_value)


# ---------------------------------------------------------------------------
# Core Pipeline Functions
# ---------------------------------------------------------------------------

def load_data(path: Path) -> pd.DataFrame:
    """Load the raw CSV into a DataFrame with informative logging."""
    logger.info(f"Loading raw data from: {path}")
    df = pd.read_csv(path, dtype=str)           # load everything as str first
    logger.info(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
    return df


def audit_raw_data(df: pd.DataFrame, report_lines: list[str]) -> None:
    """
    Perform a thorough data quality audit and append findings to report_lines.
    """
    logger.info("Running data quality audit …")

    report_lines.append("=" * 70)
    report_lines.append("AMR METADATA — DATA QUALITY AUDIT REPORT")
    report_lines.append("=" * 70)
    report_lines.append(f"Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
    report_lines.append(f"Columns: {list(df.columns)}\n")

    # Missing values
    report_lines.append("--- Missing Values ---")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    for col in df.columns:
        if missing[col] > 0:
            report_lines.append(
                f"  {col:<25} {missing[col]:>5} missing  ({missing_pct[col]:.1f}%)"
            )

    # Duplicates
    report_lines.append(f"\n--- Duplicate Rows ---")
    dupe_count = df.duplicated().sum()
    report_lines.append(f"  Exact duplicate rows: {dupe_count}")

    # Censored MIC values
    report_lines.append("\n--- Censored MIC Values (antibiotic columns) ---")
    for col in ANTIBIOTIC_COLS:
        if col not in df.columns:
            continue
        censored = df[col].dropna()
        censored = censored[censored.str.match(r"^[<>]", na=False)]
        report_lines.append(f"  {col:<20} {len(censored):>4} censored values")
        if len(censored) > 0:
            report_lines.append(f"    Examples: {list(censored.unique()[:5])}")

    # Beta.lactamase encoding
    report_lines.append("\n--- Beta.lactamase Encoding ---")
    report_lines.append(
        f"  Unique values: {df['Beta.lactamase'].value_counts().to_dict()}"
    )

    report_lines.append("\n--- Year Range ---")
    years = pd.to_numeric(df["Year"], errors="coerce").dropna()
    report_lines.append(f"  Min: {int(years.min())}  Max: {int(years.max())}")
    report_lines.append(f"  Missing: {df['Year'].isnull().sum()}")

    report_lines.append("\n--- Geographic Coverage ---")
    report_lines.append(f"  Countries : {df['Country'].nunique()} unique")
    report_lines.append(f"  Continents: {df['Continent'].dropna().unique().tolist()}")

    report_lines.append("\n")


def clean_identifiers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean Sample_ID, strip whitespace, ensure uniqueness.
    """
    logger.info("Cleaning identifiers …")
    df["Sample_ID"] = df["Sample_ID"].str.strip()
    assert df["Sample_ID"].is_unique, "Sample_ID column contains duplicates!"
    return df


def clean_year(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert Year to a nullable integer (Int16) and flag implausible values.
    Years before 1970 or after 2025 are set to NaT/NaN.
    """
    logger.info("Cleaning Year column …")
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

    implausible_mask = (df["Year"] < 1970) | (df["Year"] > 2025)
    n_implausible = implausible_mask.sum()
    if n_implausible > 0:
        logger.warning(
            f"  {n_implausible} implausible Year values detected — setting to NaN"
        )
        df.loc[implausible_mask, "Year"] = np.nan

    # Cast to pandas nullable integer
    df["Year"] = df["Year"].astype("Int16")
    return df


def clean_geography(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardise Country and Continent, strip whitespace, title-case.
    Fill the one missing Country/Continent pair where both are absent
    by marking as 'Unknown'.
    """
    logger.info("Cleaning geographic columns …")

    for col in ["Country", "Continent"]:
        df[col] = df[col].str.strip().str.title()

    # Fill isolated missing geographic data
    df["Country"]   = df["Country"].fillna("Unknown")
    df["Continent"] = df["Continent"].fillna("Unknown")

    return df


def clean_beta_lactamase(df: pd.DataFrame) -> pd.DataFrame:
    """
    Harmonise the Beta.lactamase column to a clean categorical variable.
    Adds a new column 'beta_lactamase_status' while preserving the original
    for audit purposes.
    """
    logger.info("Harmonising Beta.lactamase encoding …")
    df["beta_lactamase_status"] = df["Beta.lactamase"].apply(
        standardise_beta_lactamase
    )
    df["beta_lactamase_status"] = pd.Categorical(
        df["beta_lactamase_status"],
        categories=["Negative", "Low", "High", "Unknown"],
        ordered=False,
    )
    return df


def clean_antibiotic_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Parse the raw antibiotic phenotype columns.

    For each antibiotic column:
    - Extract the numeric MIC boundary value
    - Add a censor flag column (None / 'left' / 'right')
    - The existing *_mic numeric columns are kept as the authoritative
      numeric source; the parsed values are stored in *_mic_parsed for
      cross-validation / transparency.

    Note: For downstream modelling, prefer the existing numeric *_mic columns
    (which were pre-calculated by the original data producers) over the parsed
    values for exact measurements.
    """
    logger.info("Parsing antibiotic / censored MIC columns …")

    antibiotic_to_prefix = {
        "Azithromycin":  "azm",
        "Ciprofloxacin": "cip",
        "Ceftriaxone":   "cro",
        "Cefixime":      "cfx",
        "Tetracycline":  "tet",
        "Penicillin":    "pen",
    }

    for col, prefix in antibiotic_to_prefix.items():
        if col not in df.columns:
            continue

        parsed = df[col].apply(parse_censored_mic)
        df[f"{prefix}_mic_parsed"]  = [p[0] for p in parsed]
        df[f"{prefix}_censor_flag"] = [p[1] for p in parsed]

        # Cast censor flag to categorical for memory efficiency
        df[f"{prefix}_censor_flag"] = pd.Categorical(
            df[f"{prefix}_censor_flag"],
            categories=["left", "right"],
        )

    return df


def cast_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure all MIC, log2-MIC, and SR columns are proper float64 / Int8.
    Coerce unparseable values to NaN.
    """
    logger.info("Casting numeric columns …")

    for col in MIC_NUM_COLS + LOG2_MIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # SR columns are binary (0/1) — use nullable integer
    for col in SR_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int8")

    return df


def validate_sr_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Validate that binary SR flag columns contain only {0, 1, NaN}.
    Log a warning if unexpected values are found.
    """
    logger.info("Validating SR flag columns …")
    for col in SR_COLS:
        if col not in df.columns:
            continue
        valid_vals = {0, 1}
        actual_vals = set(df[col].dropna().astype(int).unique())
        unexpected = actual_vals - valid_vals
        if unexpected:
            logger.warning(
                f"  Column '{col}' contains unexpected values: {unexpected}"
            )
    return df


def recompute_missing_log2_mic(df: pd.DataFrame) -> pd.DataFrame:
    """
    Where log2_*_mic is NaN but the corresponding *_mic numeric value is
    present, recompute log2(MIC) to fill the gap.
    """
    logger.info("Recomputing missing log2 MIC values …")
    pairs = [
        ("azm_mic", "log2_azm_mic"),
        ("cip_mic", "log2_cip_mic"),
        ("cro_mic", "log2_cro_mic"),
        ("cfx_mic", "log2_cfx_mic"),
        ("tet_mic", "log2_tet_mic"),
        ("pen_mic", "log2_pen_mic"),
    ]
    for mic_col, log2_col in pairs:
        if mic_col not in df.columns or log2_col not in df.columns:
            continue
        missing_mask = df[log2_col].isna() & df[mic_col].notna()
        filled = df.loc[missing_mask, mic_col].apply(compute_log2_mic)
        df.loc[missing_mask, log2_col] = filled
        n_filled = missing_mask.sum()
        if n_filled:
            logger.info(f"  Recomputed {n_filled} values in '{log2_col}'")
    return df


def clean_ng_mast(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardise the NG_MAST column:
    - Strip whitespace
    - Replace '-' (dash only) with NaN — these indicate not typed
    - Preserve all other string values as-is
    """
    logger.info("Cleaning NG_MAST column …")
    df["NG_MAST"] = df["NG_MAST"].str.strip()
    df["NG_MAST"] = df["NG_MAST"].replace("-", np.nan)
    return df


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove exact duplicate rows (none expected, but included defensively).
    """
    before = len(df)
    df = df.drop_duplicates()
    removed = before - len(df)
    if removed:
        logger.warning(f"Removed {removed} exact duplicate rows")
    else:
        logger.info("No duplicate rows found")
    return df


def reorder_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reorder columns into logical groups for readability:
    1. Identifiers & metadata
    2. Geographic
    3. Phenotypic (antibiotic strings + MIC numerics)
    4. Binary SR flags
    5. Log2-MIC values
    6. Derived / cleaned columns
    """
    logger.info("Reordering columns …")

    col_order = (
        # --- Identifiers & metadata ---
        ["Sample_ID", "Year", "Group", "NG_MAST"]
        # --- Geographic ---
        + ["Country", "Continent"]
        # --- Beta-lactamase ---
        + ["Beta.lactamase", "beta_lactamase_status"]
        # --- Raw antibiotic phenotype strings ---
        + ANTIBIOTIC_COLS
        # --- Numeric MIC values ---
        + MIC_NUM_COLS
        # --- Parsed MIC values and censor flags ---
        + [c for c in df.columns if c.endswith("_mic_parsed")]
        + [c for c in df.columns if c.endswith("_censor_flag")]
        # --- Log2 MIC ---
        + LOG2_MIC_COLS
        # --- Binary SR flags ---
        + SR_COLS
    )

    # Only keep columns that actually exist
    col_order = [c for c in col_order if c in df.columns]
    # Append any remaining columns not explicitly placed
    remaining = [c for c in df.columns if c not in col_order]
    return df[col_order + remaining]


def generate_summary(df: pd.DataFrame, report_lines: list[str]) -> None:
    """Append a post-cleaning summary to the report."""
    report_lines.append("=" * 70)
    report_lines.append("POST-CLEANING SUMMARY")
    report_lines.append("=" * 70)
    report_lines.append(f"Final shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")

    report_lines.append("\n--- Remaining Missing Values ---")
    missing = df.isnull().sum()
    for col, count in missing[missing > 0].items():
        pct = count / len(df) * 100
        report_lines.append(f"  {col:<35} {count:>5} ({pct:.1f}%)")

    report_lines.append("\n--- Beta-Lactamase Status Distribution ---")
    report_lines.append(str(df["beta_lactamase_status"].value_counts()))

    report_lines.append("\n--- SR Flag Summary (counts of resistant=1) ---")
    for col in SR_COLS:
        if col in df.columns:
            n_r = df[col].sum(skipna=True)
            n_total = df[col].notna().sum()
            pct = (n_r / n_total * 100) if n_total > 0 else 0
            report_lines.append(
                f"  {col:<15} resistant: {int(n_r):>4} / {int(n_total):>4}  ({pct:.1f}%)"
            )
    report_lines.append("\n")


def save_outputs(
    df: pd.DataFrame,
    clean_path: Path,
    report_path: Path,
    report_lines: list[str],
) -> None:
    """Persist the cleaned DataFrame and the quality report."""
    # Ensure output directories exist
    clean_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.parent.mkdir(parents=True, exist_ok=True)

    logger.info(f"Saving cleaned dataset to: {clean_path}")
    df.to_csv(clean_path, index=False)

    logger.info(f"Saving quality report to:  {report_path}")
    report_path.write_text("\n".join(report_lines), encoding="utf-8")

    logger.info("Done.")


# ---------------------------------------------------------------------------
# Main Pipeline
# ---------------------------------------------------------------------------

def run_pipeline(
    raw_path: Path   = RAW_DATA_PATH,
    clean_path: Path = CLEAN_DATA_PATH,
    report_path: Path = REPORT_PATH,
) -> pd.DataFrame:
    """
    Execute the full data cleaning pipeline end-to-end.

    Parameters
    ----------
    raw_path   : Path to the raw input CSV.
    clean_path : Path where the cleaned CSV will be written.
    report_path: Path where the quality-audit text report will be written.

    Returns
    -------
    pd.DataFrame
        The fully cleaned DataFrame.
    """
    report_lines: list[str] = []

    # 1. Load
    df = load_data(raw_path)

    # 2. Audit raw data
    audit_raw_data(df, report_lines)

    # 3. Cleaning steps (order matters)
    df = remove_duplicates(df)
    df = clean_identifiers(df)
    df = clean_year(df)
    df = clean_geography(df)
    df = clean_ng_mast(df)
    df = clean_beta_lactamase(df)
    df = clean_antibiotic_columns(df)
    df = cast_numeric_columns(df)
    df = validate_sr_flags(df)
    df = recompute_missing_log2_mic(df)

    # 4. Cosmetic / structural
    df = reorder_columns(df)

    # 5. Post-clean audit
    generate_summary(df, report_lines)

    # 6. Persist
    save_outputs(df, clean_path, report_path, report_lines)

    return df


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    cleaned_df = run_pipeline()
    print(f"\n✓ Cleaning complete. Final shape: {cleaned_df.shape}")

In [ ]:
cleaned_df = run_pipeline()

In [ ]:
import os

os.listdir("data/processed")

In [ ]:
os.listdir("reports")

In [ ]:
import pandas as pd

clean_df = pd.read_csv(
    "data/processed/metadata_cleaned.csv"
)

clean_df.head()

In [ ]:
clean_df.isnull().sum().sort_values(
    ascending=False
)

In [ ]:
from google.colab import files

files.download(
    "data/processed/metadata_cleaned.csv"
)

In [ ]:
files.download(
    "reports/data_quality_report.txt"
)